In [29]:
import numpy as np
import pandas as pd

In [30]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

In [31]:
df = pd.read_csv("Customer-Churn.csv")


In [32]:
df = df.drop("customerID", axis=1)

In [33]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [34]:
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
df["Churn"]

0        No
1        No
2       Yes
3        No
4       Yes
       ... 
7038     No
7039     No
7040     No
7041    Yes
7042     No
Name: Churn, Length: 7043, dtype: object

In [35]:
df["Churn"] = df["Churn"].map({"Yes":1,"No":0})
df["Churn"]

0       0
1       0
2       1
3       0
4       1
       ..
7038    0
7039    0
7040    0
7041    1
7042    0
Name: Churn, Length: 7043, dtype: int64

In [36]:
x = df.drop("Churn",axis=1)

In [37]:
x = df.drop("Churn",axis=1)
y = df["Churn"]
y

0       0
1       0
2       1
3       0
4       1
       ..
7038    0
7039    0
7040    0
7041    1
7042    0
Name: Churn, Length: 7043, dtype: int64

In [38]:
cat_cols = x.select_dtypes(include="object").columns
num_cols = x.select_dtypes(exclude="object").columns

In [39]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

In [40]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


In [41]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline,num_cols),
    ("cat", categorical_pipeline, cat_cols)
])

In [42]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size = 0.2,
    stratify=y,
    random_state=42
)

In [43]:
x_train_prep = preprocessor.fit_transform(x_train)
x_test_prep = preprocessor.transform(x_test)

In [44]:
selector_model = RandomForestClassifier(
  n_estimators=300,
    random_state=42
)

In [45]:
selector_model.fit(x_train_prep, y_train)
selector = SelectFromModel(
    selector_model,
    threshold="median",
    prefit=True
)

In [46]:
x_train_sel = selector.transform(x_train_prep)
x_test_sel = selector.transform(x_test_prep)

In [47]:
print("Original features:", x_train_prep.shape[1])
print("Selected features:", x_train_sel.shape[1])

Original features: 45
Selected features: 23


In [48]:
# 9. Final Random Forest Model
# ==============================================

model = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    class_weight="balanced",
    random_state=42
)

In [49]:
model.fit(x_train_sel, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=500,
                       random_state=42)

In [50]:
pred = model.predict(x_test_sel)
prob = model.predict_proba(x_test_sel)[:, 1]

In [51]:
print("\nConfusion Matrix")
print(confusion_matrix(y_test, pred))

print("\nClassification Report")
print(classification_report(y_test, pred))

print("\nROC AUC:", roc_auc_score(y_test, prob))




Confusion Matrix
[[830 205]
 [117 257]]

Classification Report
              precision    recall  f1-score   support

           0       0.88      0.80      0.84      1035
           1       0.56      0.69      0.61       374

    accuracy                           0.77      1409
   macro avg       0.72      0.74      0.73      1409
weighted avg       0.79      0.77      0.78      1409


ROC AUC: 0.8391588519465758


In [52]:
ohe = preprocessor.named_transformers_["cat"]["encoder"]
cat_feature_names = ohe.get_feature_names_out(cat_cols)

In [53]:
all_feature_names = list(num_cols) + list(cat_feature_names)

In [54]:
selected_mask = selector.get_support()

In [55]:
selected_features = np.array(all_feature_names)[selected_mask]

In [56]:
print("\nSelected Feature Names:")
for f in selected_features:
    print(f)


Selected Feature Names:
SeniorCitizen
tenure
MonthlyCharges
TotalCharges
gender_Female
gender_Male
Partner_No
Partner_Yes
Dependents_Yes
MultipleLines_No
MultipleLines_Yes
InternetService_Fiber optic
OnlineSecurity_No
OnlineBackup_No
OnlineBackup_Yes
DeviceProtection_No
TechSupport_No
Contract_Month-to-month
Contract_Two year
PaperlessBilling_No
PaperlessBilling_Yes
PaymentMethod_Credit card (automatic)
PaymentMethod_Electronic check
